In [1]:
# !pip install jsonschema
import json
from jsonschema import validate
from constants import ValidationError, ErrorType, assert_all_logs

from engine import Engine
import os

data_file = 'data.json'
schema_file = 'schema.json'
data_enum_error_file = 'data_enum_error.json'
data_type_error_file = 'data_type_error.json'
data_no_schema_error_file = 'data_no_schema_error.json'
data_unclosed_error_file = 'data_unclosed_error.json'

In [2]:
json_data = json.load(open(data_file))
json_schema = json.load(open(schema_file))

# Validate the JSON data against the schema using jsonschema library
# jsonschema requires valid python dict for both data and schema
validate(instance=json_data, schema=json_schema)

In [3]:
def verify_json(schema=schema_file, data=data_file, max_depth = None):
    engine = Engine(schema=schema, target=data, max_depth=max_depth)
    log = engine.run()
    return log

In [4]:
# valid json data verification
verify_json(data=data_file)

[]

In [5]:
# type error verification
log = verify_json(data=data_type_error_file)
target_errors = {
    'top_object.a': ValidationError(ErrorType.BAD_VALUE, {'value': 'c', 'rule': 'type(number)'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'a'})
}

# print(log)
assert_all_logs(log, target_errors)

In [6]:
# unclosed error verification
log = verify_json(data=data_unclosed_error_file)

target_errors = {
    'top_object.d': ValidationError(ErrorType.UNCLOSED),
    'top_object': ValidationError(ErrorType.UNCLOSED)
}

assert_all_logs(log, target_errors)

In [7]:
# maximum stack depth verification
log = verify_json(data=data_file, max_depth=2)

target_errors = {
    'circuit_breaker': ValidationError(ErrorType.DEPTH_ERROR, {'depth': 2})
}

assert_all_logs(log, target_errors)

In [ ]:
# enumeration error verification
log = verify_json(data=data_enum_error_file)
target_errors = {
    'top_object.d[2]': ValidationError(ErrorType.BAD_VALUE, {'value': 5, 'rule': 'enum([3, 4])'}),
    'top_object.d': ValidationError(ErrorType.INCOMPLETE, {'value': 2}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'd'})
}

assert_all_logs(log, target_errors)

[invalid json item: path(top_object.d[2])
   node info: 5
     <BAD VALUE>: value(5) violates schema[enum([3, 4])],
 invalid json item: path(top_object.d)
   node info: {0: valid, 1: valid, 2: invalid}
     <INCOMPLETE>: child(2) is not valid,
 invalid json item: path(top_object)
   node info: {'a': valid, 'b': valid, 'd': invalid}
     <INCOMPLETE>: child(d) is not valid]

In [ ]:
# unexpected json object verification
log = verify_json(data=data_no_schema_error_file)

target_errors = {
    'top_object.b.e': ValidationError(ErrorType.UNEXPECTED, {'value': 'extra'}),
    'top_object.b': ValidationError(ErrorType.INCOMPLETE, {'value': 'e'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'b'})
}

assert_all_logs(log, target_errors)

{'value': 'extra'}
{'value': 'e'}
{'value': 'b'}
